# Dannyl MOUHEB – Modèle churn ClientCo – 1. Modèle mensuel (chiffres de la présentation)

**Ce que fait ce notebook**
1. Lit le CSV `data/dataset_sales_2026.csv` (ou le cache `dannyl_modele_churn/cache/agregats.pkl`).
2. Prend une **photo** du portefeuille le 1er de chaque mois (oct. 2023 → mars 2024) : pour chaque client régulier, 28 **features** calculées avant la date et les **cibles** observées après.
3. Sépare les clients : 80 % pour apprendre (photos d'oct. 2023 à janv. 2024), 20 % jamais vus pour tester (photo du 1er mars 2024).
4. Compare 3 modèles (règle simple, régression logistique, XGBoost) sur 3 cibles (départ total, forte baisse, les deux).
5. Évalue la **liste d'appels** (clients classés par probabilité × CA, coupés à la capacité des commerciaux) et valorise à l'échelle du groupe.

**Cibles** : départ total = aucun achat pendant 3 × le rythme habituel du client (entre 90 et 180 jours) ; forte baisse = CA du trimestre < 50 % du même trimestre l'an dernier et du trimestre moyen. Le détail de chaque feature est dans `dannyl_commun.py` (`DESCRIPTION_VARIABLES`).

**Sortie principale** : `dannyl_modele_churn/resultats/mensuel/resultats.json` (à envoyer pour mettre à jour les slides).

In [ ]:
# --- Mise en place (ne pas modifier) -------------------------------------------------------------
# Le notebook se place à la racine du dépôt (où se trouve data/) et rend importables les modules
# du dossier dannyl_modele_churn.
import os, sys, json
import pandas as pd
from IPython.display import Image, display
if os.path.basename(os.getcwd()) == "dannyl_modele_churn":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("dannyl_modele_churn"))
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
RES = os.path.join("dannyl_modele_churn", "resultats")
print("Dossier de travail :", os.getcwd())

## Paramètres
`CONTACTS_PAR_MOIS` : contacts anti-churn par commercial et par mois (hypothèse : 10 % du temps = 2 visites + 18 appels).

In [ ]:
CONTACTS_PAR_MOIS = 20
PART_CLIENTS_TEST = 0.20

## Lancement (10 à 30 min au premier lancement pour lire les 4 Go, puis quelques minutes)

In [ ]:
import importlib, dannyl_1_modele_mensuel as script
importlib.reload(script)
sys.argv = ["dannyl_1_modele_mensuel"] + ["--contacts-par-mois", str(CONTACTS_PAR_MOIS), "--part-clients-test", str(PART_CLIENTS_TEST)]
script.main()

## Résultats
### Représentativité du dataset et capacité commerciale

In [ ]:
R = json.load(open(os.path.join(RES, "mensuel", "resultats.json"), encoding="utf-8"))
display(pd.Series(R["dataset"], name="dataset").to_frame())
display(pd.Series(R["capacite"], name="capacité").to_frame())

### Combien de clients partent ? (photo du 1er mars 2024, tous les clients réguliers)

In [ ]:
display(pd.DataFrame(R["churn_descriptif"]).T)
display(pd.Series(R["silence_vs_rythme"], name="part des clients selon silence / rythme habituel").to_frame())

### Performance des modèles (clients et période jamais vus)
- **perte_captee_k** : part du CA à risque captée par la liste d'appels (indicateur principal)
- **precision_k** : sur 100 clients appelés, combien partent vraiment ; **rappel_k_gros** : part des gros clients qui partent présents dans la liste
- **auc / pr_auc / brier** : qualité du classement et des probabilités

In [ ]:
lignes = []
for cible, d in R["metriques"].items():
    for modele, m in d.items():
        lignes.append(dict(cible=cible, modele=modele, **{k: v for k, v in m.items() if k not in ("confusion", "perte_captee_k_meur")}))
display(pd.DataFrame(lignes).set_index(["cible", "modele"]))
print("Modèle retenu :", R["modele_retenu"])
print("Intervalle de confiance à 90 % du CA à risque capté :", R["intervalle_90_perte_captee"])

### Matrice de confusion de la liste d'appels (modèle retenu, cible principale)

In [ ]:
c = R["metriques"]["churn"][R["modele_retenu"]]["confusion"]
display(pd.DataFrame([[c["vrais_positifs"], c["faux_positifs"]], [c["departs_manques"], c["vrais_negatifs"]]],
                     index=["Dans la liste", "Hors liste"], columns=["Partent vraiment", "Restent"]))

### Courbe de gain et calibration

In [ ]:
display(Image(os.path.join(RES, "mensuel", "courbe_gain.png")))
if "calibration" in R:
    display(pd.DataFrame(R["calibration"]))

### Lecture métier, valeur et features les plus utiles

In [ ]:
display(pd.Series(R["lecture"], name="lecture").to_frame())
display(pd.Series(R["valeur"], name="valeur (M€ / an, groupe)").to_frame())
display(pd.Series(R.get("importance_variables", {}), name="importance (permutation, PR-AUC)").to_frame())

### Exemple de liste d'appels mensuelle

In [ ]:
display(pd.read_csv(os.path.join(RES, "mensuel", "liste_appels_exemple.csv"), sep=";").head(20))